In [78]:
#-- Import libraries. --#
import json
import folium
import pandas as pd
from folium.plugins import Search


In [79]:
file_path = '../Processed/roads_final_cleaned.csv'
try:
    df = pd.read_csv(file_path)
except FileNotFoundError:
    print(f"Error: Cannot find file '{file_path}'.")
    exit()

In [80]:
df['tfm_id'] = df['tfm_id'].astype(str)
df['road_name'] = df['declared_r'].fillna('Unknown Road')
boro_df = df[(df['x'] > 145.02) & (df['x'] < 145.12) & 
             (df['y'] > -37.85) & (df['y'] < -37.78)].copy()

In [81]:
m = folium.Map(location=[-37.815, 145.07], zoom_start=15, tiles=None)

In [82]:
folium.TileLayer(tiles='https://mt1.google.com/vt/lyrs=s&x={x}&y={y}&z={z}', 
                 attr='Google', name='Satellite View', overlay=False).add_to(m)
folium.TileLayer(tiles='https://{s}.basemaps.cartocdn.com/dark_all/{z}/{x}/{y}{r}.png', 
                 attr='CartoDB', name='Dark Mode (Pro)', overlay=False).add_to(m)
folium.TileLayer(tiles='https://{s}.basemaps.cartocdn.com/light_all/{z}/{x}/{y}{r}.png', 
                 attr='CartoDB', name='Light Mode (Clean)', overlay=False).add_to(m)

In [83]:
features = []
for idx, row in boro_df.iterrows():
    features.append({
        "type": "Feature",
        "geometry": {"type": "Point", "coordinates": [row['x'], row['y']]},
        "properties": {
            # Key này dùng để search cả ID và Tên đường
            "search_val": f"{row['tfm_id']} - {row['road_name']}",
            "road": row['road_name'],
            "lat": row['y'],
            "lon": row['x']
        }
    })
stations_geojson = {"type": "FeatureCollection", "features": features}

In [84]:
station_layer = folium.GeoJson(
    stations_geojson,
    name="Traffic Stations (Purple Nodes)",
    marker=folium.CircleMarker(
        radius=5, # Tăng nhẹ kích thước để dễ nhìn hơn khi bỏ đường nối
        color='#00FFFF', # Viền Cyan nổi bật
        weight=0.5,
        fill=True, 
        fill_color='#6A0DAD', # Màu tím
        fill_opacity=0.9
    ),
    tooltip=folium.GeoJsonTooltip(fields=['road'], aliases=['Station:']),
    # ĐÁP ỨNG YÊU CẦU: Hiển thị đầy đủ Road Name, Latitude, Longitude khi click
    popup=folium.GeoJsonPopup(
        fields=['road', 'lat', 'lon'],
        aliases=['📍 Road:', '🌐 Latitude:', '🌐 Longitude:']
    )
).add_to(m)

In [85]:
Search(
    layer=station_layer,
    geom_type='Point',
    placeholder='Search ID or Road Name...',
    collapsed=False, # Hiển thị luôn ô search
    search_label='search_val', # Dùng key gộp để search
    search_zoom=18 # Zoom vào sẽ thấy vòng tròn đỏ bao quanh trạm (mặc định của plugin)
).add_to(m)

In [86]:
table_html = f"""
<div style="position: fixed; bottom: 20px; left: 10px; width: 320px; height: 280px; 
            z-index:9999; background-color: rgba(255, 255, 255, 0.9); padding: 10px;
            border-radius: 8px; border: 2px solid #6A0DAD; overflow-y: auto; font-family: sans-serif;
            box-shadow: 3px 3px 5px rgba(0,0,0,0.3);">
    <h4 style="margin: 0 0 10px 0; color: #6A0DAD; text-align: center;">📍 Station List</h4>
    <table style="width:100%; border-collapse: collapse; font-size: 11px;">
        <thead>
            <tr style="background-color: #6A0DAD; color: white;">
                <th style="padding: 6px; text-align: left; border-bottom: 2px solid #fff;">ID</th>
                <th style="padding: 6px; text-align: left; border-bottom: 2px solid #fff;">Station Name</th>
            </tr>
        </thead>
        <tbody>
"""
for idx, row in boro_df.iterrows():
    # Thêm hiệu ứng hover và click để zoom vào trạm
    table_html += f"""
            <tr style="cursor: pointer; border-bottom: 1px solid #eee;" 
                onmouseover="this.style.backgroundColor='#f0faff'" 
                onmouseout="this.style.backgroundColor='transparent'" 
                onclick="m.flyTo([{row['y']}, {row['x']}], 18)">
                <td style="padding: 6px;">{row['tfm_id']}</td>
                <td style="padding: 6px;"><b>{row['road_name']}</b></td>
            </tr>
    """
table_html += "</tbody></table></div>"

m.get_root().html.add_child(folium.Element(table_html))

In [87]:
folium.LayerControl(collapsed=False).add_to(m)

In [88]:
output_file = 'Boroondara_Clean_Dashboard.html'
m.save(output_file)
print(f"Xong! Đã tạo file '{output_file}'. Bản đồ đã gọn gàng.")

Xong! Đã tạo file 'Boroondara_Clean_Dashboard.html'. Bản đồ đã gọn gàng.
